# Python File Handling

File handling allows programs to read, write, update, and manage persistent data on disk rather than storing it transiently in volatile memory.


### 1. Basics & The `with` Statement (Context Manager)

**Explanation**: The open() function opens a file and returns a file object. The recommended approach is using the 'with' statement, which ensures the file is automatically and reliably closed as soon as the block exits, even if exceptions occur.

**Syntax**:
```python
with open(filename, mode, encoding='utf-8') as file:
    # file operations
```

In [1]:
# 1. Writing and Reading with the 'with' Statement
filename = "employee.txt"

# Writing to a file (automatically closed upon exiting the block)
with open(filename, "w", encoding="utf-8") as file:
    file.write("Employee Name: Raj\n")
    file.write("Department: IT\n")
    file.write("Salary: 50000\n")

# Reading the file
with open(filename, "r", encoding="utf-8") as file:
    content = file.read()
    print("--- File Content ---")
    print(content.strip())

# Checking that the file is safely closed
print(f"Is file closed? {file.closed}")

--- File Content ---
Employee Name: Raj
Department: IT
Salary: 50000
Is file closed? True


---
## 2. File Opening Modes

Python supports several opening modes determining access permissions, pointer placement, and whether existing contents are truncated:

| Mode | Operation | If File Exists? | If File Doesn't Exist? | Pointer Placement |
| :--- | :--- | :--- | :--- | :--- |
| `'r'` | Read only | Reads content | Raises `FileNotFoundError` | Beginning |
| `'w'` | Write only | **Truncates (erases)** content | Creates new file | Beginning |
| `'a'` | Append only | Retains content | Creates new file | End |
| `'x'` | Exclusive create | Raises `FileExistsError` | Creates new file | Beginning |
| `'r+'`| Read + Write | Retains content | Raises `FileNotFoundError` | Beginning |
| `'w+'`| Write + Read | **Truncates (erases)** content | Creates new file | Beginning |
| `'a+'`| Append + Read | Retains content | Creates new file | End |

In [2]:
# 2. File Modes Demonstration
# Mode 'a': Append without erasing existing content
with open("employee.txt", "a", encoding="utf-8") as file:
    file.write("Location: Bangalore\n")

# Mode 'x': Exclusive creation (fails safely if file already exists)
try:
    with open("employee.txt", "x", encoding="utf-8") as file:
        file.write("This will fail because file exists.")
except FileExistsError:
    print("Mode 'x' caught: File already exists; overwriting prevented!")

# Mode 'r+': Read and update in-place without truncation
with open("employee.txt", "r+", encoding="utf-8") as file:
    data = file.read()
    file.write("Status: Active\n")

# Verify updated content
with open("employee.txt", "r", encoding="utf-8") as file:
    print("\n--- Updated employee.txt ---")
    print(file.read().strip())

Mode 'x' caught: File already exists; overwriting prevented!

--- Updated employee.txt ---
Employee Name: Raj
Department: IT
Salary: 50000
Location: Bangalore
Status: Active


### 3. File Pointer Mechanics: `tell()` and `seek()`

**Explanation**: A file pointer tracks the current byte offset where reading or writing takes place. `file.tell()` reports the current offset, while `file.seek(offset, whence)` navigates the pointer to a designated position.

**Syntax**:
```python
current_pos = file.tell()  # Returns byte position
file.seek(0)               # Rewinds pointer to start of file
```

In [3]:
# 3. tell() and seek() Example
demo_file = "pointer_demo.txt"

with open(demo_file, "w+", encoding="utf-8") as file:
    file.write("Hello Python Programming")
    
    # Check pointer position after writing
    print(f"Pointer position after write: {file.tell()}")
    
    # Rewind pointer back to start to read
    file.seek(0)
    print(f"Pointer position after seek(0): {file.tell()}")
    print("Read first 5 chars:", file.read(5))   # 'Hello'
    print(f"Pointer position after reading: {file.tell()}")

    # Seek to offset 6 and overwrite 'Python' with 'Golang'
    file.seek(6)
    file.write("Golang")
    
    # Read entire modified content
    file.seek(0)
    print("Modified content:", file.read())

Pointer position after write: 24
Pointer position after seek(0): 0
Read first 5 chars: Hello
Pointer position after reading: 5
Modified content: Hello Golang Programming


### 4. Essential File Methods

**Explanation**: Methods for input, output, and inspection: `read()`, `readline()`, `readlines()`, `write()`, `writelines()`, `truncate()`, and `flush()`.

**Syntax**:
```python
# read(n) / readline() / readlines()
# write(str) / writelines(list)
# truncate(size) / flush()
```

In [4]:
# 4. Built-in File Methods Demo
test_file = "methods_demo.txt"

# writelines(): writes iterable of strings without implicit newlines
lines = ["Line 1: Python\n", "Line 2: Java\n", "Line 3: Rust\n"]
with open(test_file, "w", encoding="utf-8") as f:
    f.writelines(lines)
    f.flush()  # Manually flush buffer to disk

In [5]:
test_file = "methods_demo.txt"
# readline() vs readlines()
with open(test_file, "r", encoding="utf-8") as f:
    first_line = f.readline()
    remaining_lines = f.readlines()
    print("readline()  :", first_line.strip())
    print("readlines() :", [l.strip() for l in remaining_lines])

readline()  : Line 1: Python
readlines() : ['Line 2: Java', 'Line 3: Rust']


In [6]:
test_file = "methods_demo.txt"
# truncate(size): resizes file to specific byte size
with open(test_file, "r+", encoding="utf-8") as f:
    f.truncate(6)  # Keep only the first 6 bytes
    f.seek(0)
    print("Truncated content:", f.read())

Truncated content: Line 1


### 5. Practical Plain Text File Processing

**Explanation**: Efficient text file parsing: line-by-line streaming using standard file iteration, whitespace stripping (`strip()`), word searching, and aggregate counting.

**Syntax**:
```python
for line in file:
    processed = line.strip()
```

In [7]:
# 5. Text Processing: Searching & Counting Lines, Words, Characters
data_file = "sample_text.txt"

# Prepare sample file
sample_content = """Python is a high-level programming language.
Python is widely used in data science, AI, and backend web development.
File handling in Python is simple and expressive.
"""

with open(data_file, "w", encoding="utf-8") as f:
    f.write(sample_content)

# Search & Metrics calculation (single pass without loading full file to RAM)
search_keyword = "Python"
total_lines, total_words, total_chars = 0, 0, 0

print("--- Keyword Search Results ---")
with open(data_file, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, start=1):
        total_lines += 1
        total_words += len(line.split())
        total_chars += len(line)
        if search_keyword in line:
            print(f"Line {line_num}: {line.strip()}")

print(f"\nMetrics -> Lines: {total_lines}, Words: {total_words}, Characters: {total_chars}")

--- Keyword Search Results ---
Line 1: Python is a high-level programming language.
Line 2: Python is widely used in data science, AI, and backend web development.
Line 3: File handling in Python is simple and expressive.

Metrics -> Lines: 3, Words: 26, Characters: 167


---
## 6. JSON File Handling (`json` module)

JSON (**J**ava**S**cript **O**bject **N**otation) is the standard lightweight interchange format for APIs, configs, and structured data.

### Four Core JSON Methods:
| Method | Operates On | Input | Output | Purpose |
| :--- | :--- | :--- | :--- | :--- |
| `json.dumps()` | In-Memory String | Python object (`dict`, `list`) | JSON formatted `str` | Convert Python data to JSON string |
| `json.loads()` | In-Memory String | JSON formatted `str` | Python object | Parse JSON string into Python object |
| `json.dump()` | File System | Python object + file handle | Writes directly to disk | Serialize and save Python data to file |
| `json.load()` | File System | File handle | Python object | Read and deserialize JSON file from disk |

In [8]:
# 6. JSON Serialization & Deserialization Example
import json

# 1. In-Memory: dumps() and loads()
user_profile = {
    "id": 101,
    "name": "Raj",
    "is_active": True,
    "skills": ["Python", "SQL"]
}

In [9]:
# dumps(): Python dict -> JSON String
json_string = json.dumps(user_profile, indent=2)
print("--- json.dumps() Output ---")
print(json_string)

--- json.dumps() Output ---
{
  "id": 101,
  "name": "Raj",
  "is_active": true,
  "skills": [
    "Python",
    "SQL"
  ]
}


In [10]:
# loads(): JSON String -> Python dict
parsed_dict = json.loads(json_string)
print("\nParsed type:", type(parsed_dict))
print("User name  :", parsed_dict["name"])


Parsed type: <class 'dict'>
User name  : Raj


In [11]:
# 2. File-Based: dump() and load()
json_file = "user.json"

# dump(): Write directly to file
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(user_profile, f, indent=4)

# load(): Read directly from file
with open(json_file, "r", encoding="utf-8") as f:
    loaded_data = json.load(f)

print("\nLoaded from file:", loaded_data)


Loaded from file: {'id': 101, 'name': 'Raj', 'is_active': True, 'skills': ['Python', 'SQL']}


### 7. Real-World JSON CRUD Operations

**Explanation**: JSON files store structured data as a single document. Updating records involves reading the data into memory, modifying the Python collection, and writing the updated state back to disk.

**Syntax**:
```python
# 1. Read: data = json.load(f)
# 2. Mutate: data.append(new_item)
# 3. Write: json.dump(data, f, indent=4)
```

In [12]:
# 7. Complete JSON CRUD Operations (Create, Read, Update, Delete, Filter)
import json

db_file = "employees_db.json"

# Initial seed data
initial_employees = [
    {"id": 101, "name": "Raj", "department": "IT", "salary": 50000},
    {"id": 102, "name": "Kumar", "department": "HR", "salary": 40000},
    {"id": 103, "name": "John", "department": "IT", "salary": 65000}
]

with open(db_file, "w", encoding="utf-8") as f:
    json.dump(initial_employees, f, indent=4)


In [13]:

# 1. CREATE: Append a new employee
with open(db_file, "r", encoding="utf-8") as f:
    employees = json.load(f)

employees.append({"id": 104, "name": "Anita", "department": "Finance", "salary": 55000})

with open(db_file, "w", encoding="utf-8") as f:
    json.dump(employees, f, indent=4)

### 8. Advanced JSON: Custom Serialization & Formatting

**Explanation**: Handling objects that are not natively serializable (like datetime, UUID, sets) using custom serializer functions, sorting keys, and compact minification.

**Syntax**:
```python
json.dumps(data, default=custom_serializer, indent=4, sort_keys=True)
```

In [14]:
# 8. Advanced JSON Serialization & Formatting
import json
from datetime import datetime

class UserSession:
    def __init__(self, user_id, roles):
        self.user_id = user_id
        self.roles = roles
        self.login_time = datetime.now()

session = UserSession(9901, {"admin", "developer"})

# Custom serializer function for unsupported types
def custom_json_serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    if isinstance(obj, set):
        return list(obj)
    raise TypeError(f"Type {type(obj).__name__} is not JSON serializable")

payload = {
    "session_id": "sess_abc123",
    "user_id": session.user_id,
    "roles": session.roles,
    "login_time": session.login_time
}

# Serialize with custom encoder, key sorting, and UTF-8 characters preserved
pretty_json = json.dumps(payload, default=custom_json_serializer, indent=4, sort_keys=True)
print("--- Pretty & Custom Encoded JSON ---")
print(pretty_json)

# Minified / Compact JSON for efficient network transmission
compact_json = json.dumps(payload, default=custom_json_serializer, separators=(',', ':'))
print("\n--- Compact / Minified JSON ---")
print(compact_json)

--- Pretty & Custom Encoded JSON ---
{
    "login_time": "2026-10-30T11:19:09.419717",
    "roles": [
        "admin",
        "developer"
    ],
    "session_id": "sess_abc123",
    "user_id": 9901
}

--- Compact / Minified JSON ---
{"session_id":"sess_abc123","user_id":9901,"roles":["admin","developer"],"login_time":"2026-10-30T11:19:09.419717"}


### 9. High-Performance Large File Processing

**Explanation**: Loading multi-gigabyte files entirely via read() causes high memory overhead or Out-Of-Memory (OOM) crashes. Stream line-by-line, process in fixed-size byte chunks, or use Python generator functions for lazy evaluation.

**Syntax**:
```python
# 1. Line streaming: for line in file:
# 2. Chunking: while chunk := file.read(CHUNK_SIZE):
# 3. Generator pipeline: yield line
```

In [15]:
# 9. Large File Processing Techniques
log_file = "server_activity.log"

# Generate mock server log entries
with open(log_file, "w", encoding="utf-8") as f:
    for i in range(1, 201):
        status = "ERROR" if i % 15 == 0 else ("WARNING" if i % 7 == 0 else "INFO")
        f.write(f"2026-10-30 10:00:{i%60:02d} [{status}] Server ping response from node_{i}\n")

In [16]:
# Technique 1: Memory-efficient Generator Pipeline (Lazy Evaluation)
def stream_error_logs(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if "[ERROR]" in line:
                yield line.strip()

print("--- Streamed ERROR Logs via Generator ---")
for err in stream_error_logs(log_file):
    print("Filtered:", err)


--- Streamed ERROR Logs via Generator ---
Filtered: 2026-10-30 10:00:15 [ERROR] Server ping response from node_15
Filtered: 2026-10-30 10:00:30 [ERROR] Server ping response from node_30
Filtered: 2026-10-30 10:00:45 [ERROR] Server ping response from node_45
Filtered: 2026-10-30 10:00:00 [ERROR] Server ping response from node_60
Filtered: 2026-10-30 10:00:15 [ERROR] Server ping response from node_75
Filtered: 2026-10-30 10:00:30 [ERROR] Server ping response from node_90
Filtered: 2026-10-30 10:00:45 [ERROR] Server ping response from node_105
Filtered: 2026-10-30 10:00:00 [ERROR] Server ping response from node_120
Filtered: 2026-10-30 10:00:15 [ERROR] Server ping response from node_135
Filtered: 2026-10-30 10:00:30 [ERROR] Server ping response from node_150
Filtered: 2026-10-30 10:00:45 [ERROR] Server ping response from node_165
Filtered: 2026-10-30 10:00:00 [ERROR] Server ping response from node_180
Filtered: 2026-10-30 10:00:15 [ERROR] Server ping response from node_195


In [17]:

# Technique 2: Fixed-size Chunk Processing (Ideal for arbitrary/binary data)
chunk_size = 128  # bytes
total_bytes_read = 0

with open(log_file, "rb") as f:
    while True:
        chunk = f.read(chunk_size)
        if not chunk:
            break
        total_bytes_read += len(chunk)

print(f"\nProcessed {total_bytes_read} total bytes in chunks of {chunk_size} bytes.")


Processed 12386 total bytes in chunks of 128 bytes.


### 10. Streaming Datasets with JSON Lines (`.jsonl`)

**Explanation**: JSON Lines (NDJSON) stores one standalone JSON document per line. Unlike monolithic JSON arrays, JSONL files can be appended to in O(1) time and streamed line-by-line with minimal RAM overhead.

**Syntax**:
```python
# Write: file.write(json.dumps(record) + '\n')
# Read: for line in file: record = json.loads(line)
```

In [18]:
# 10. JSON Lines (JSONL) Streaming
import json

jsonl_file = "events.jsonl"

# 1. Writing records line by line
events = [
    {"event_id": 1, "action": "login", "user": "alice"},
    {"event_id": 2, "action": "view_cart", "user": "bob"},
    {"event_id": 3, "action": "checkout", "user": "alice"}
]

with open(jsonl_file, "w", encoding="utf-8") as f:
    for ev in events:
        f.write(json.dumps(ev) + "\n")

In [19]:
# 2. O(1) Fast Append (impossible with regular JSON arrays without rewriting)
new_event = {"event_id": 4, "action": "logout", "user": "alice"}
with open(jsonl_file, "a", encoding="utf-8") as f:
    f.write(json.dumps(new_event) + "\n")

In [20]:
# 3. Stream reading line by line without memory bloat
print("--- Reading JSONL Events ---")
with open(jsonl_file, "r", encoding="utf-8") as f:
    for line in f:
        event_obj = json.loads(line)
        print(f"Event #{event_obj['event_id']}: {event_obj['action']} by {event_obj['user']}")

--- Reading JSONL Events ---
Event #1: login by alice
Event #2: view_cart by bob
Event #3: checkout by alice
Event #4: logout by alice


### 11. Modern Filesystem Management with `pathlib`

**Explanation**: Python's `pathlib` module offers an object-oriented, cross-platform interface for filesystem paths, directory traversal, extension matching (`glob`), and file metadata inspection.

**Syntax**:
```python
from pathlib import Path
p = Path('data/subdir')
p.mkdir(parents=True, exist_ok=True)
```

In [21]:
# 11. pathlib Modern Path Management
from pathlib import Path

# Create directory tree
output_dir = Path("data_store/logs")
output_dir.mkdir(parents=True, exist_ok=True)

# Path concatenation using the '/' operator
config_path = output_dir / "app_config.txt"
config_path.write_text("DEBUG=True\nVERSION=1.0.0", encoding="utf-8")

# Inspecting properties
print(f"Path exists?     : {config_path.exists()}")
print(f"Is file?         : {config_path.is_file()}")
print(f"Filename         : {config_path.name}")
print(f"File suffix/ext  : {config_path.suffix}")
print(f"Parent directory : {config_path.parent}")
print(f"File byte size   : {config_path.stat().st_size} bytes")

# Read text directly via pathlib convenience method
print("Direct read text :\n", config_path.read_text(encoding="utf-8"))

# Glob directory searching
print("\nMatching files in current folder:")
for txt_file in Path(".").glob("*.txt"):
    print(" - Found:", txt_file.name)

Path exists?     : True
Is file?         : True
Filename         : app_config.txt
File suffix/ext  : .txt
Parent directory : data_store/logs
File byte size   : 24 bytes
Direct read text :
 DEBUG=True
VERSION=1.0.0

Matching files in current folder:
 - Found: sample_text.txt
 - Found: pointer_demo.txt
 - Found: methods_demo.txt
 - Found: employee.txt


### 12. Production-Grade Safe File Updates: Atomic Writing

**Explanation**: Writing directly with mode 'w' truncates the file immediately; a crash mid-write results in data loss or corrupted files. Atomic writing writes to a temporary sibling file first and swaps it via `os.replace()` in a single atomic filesystem step.

**Syntax**:
```python
temp_file = target.with_name(target.name + '.tmp')
# write & flush temp_file
os.replace(temp_file, target)
```

In [22]:
# 12. Atomic File Writing Demonstration
import json
import os
from pathlib import Path

def atomic_save_json(data, target_filename):
    target = Path(target_filename)
    # Temporary sibling file on the same filesystem
    temp_target = target.with_name(f"{target.name}.tmp")
    
    with open(temp_target, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)
        f.flush()
        # Force physical write to storage device
        os.fsync(f.fileno())
    
    # Atomic replace: readers see either the old file or the new file, never a partial write
    os.replace(temp_target, target)
    print(f"Successfully saved {target_filename} atomically.")

sample_app_state = {
    "status": "HEALTHY",
    "uptime_seconds": 3600,
    "active_connections": 42
}

atomic_save_json(sample_app_state, "system_state.json")

# Verify written data
with open("system_state.json", "r", encoding="utf-8") as f:
    print("Verified Content:", json.load(f))

Successfully saved system_state.json atomically.
Verified Content: {'status': 'HEALTHY', 'uptime_seconds': 3600, 'active_connections': 42}


In [23]:
import os
import shutil

files = ["employee.txt", "employees_db.json", "events.jsonl", "methods_demo.txt", "pointer_demo.txt", "sample_text.txt", "server_activity.log", "system_state.json", "user.json"]
folder = "data_store"

for file in files:
    if os.path.exists(file):
        os.remove(file)

if os.path.exists(folder):
    shutil.rmtree(folder)